# ⚡ spark02 — RDD Core
### Section 18 · 14 Videos · Chandra Sekhar Sahu
**Complete this notebook while watching Section 18 videos**

---
```
Videos in Section 18:
01. What is Spark RDD
02. How Spark Reads Data
03. Spark Read Data and Partitioning
04. RDD Operations Part 1
05. RDD Operations Part 2
06. Narrow vs Wide Transformations
07. Jobs, Stages and Tasks in Spark UI
08. GroupByKey vs ReduceByKey Part 1
09. GroupByKey vs ReduceByKey Part 2
10. Repartition vs Coalesce
11. Higher Level APIs — DataFrame
12. Spark SQL
13. Spark Streaming Overview
14. Section Summary
```


---
## 📌 BUILD 1 — What is an RDD?
*Watch Video 01 first*

### Fill in the blanks:

**RDD stands for:** _______________

**R = ** Resilient 
**D = ** Distributed 
**D = ** DataSysytem

---

**3 Key Properties of RDD:**

1. _______________
2. _______________
3. _______________

---

**RDD vs DataFrame — main difference:**

```
RDD         → works with distributed data
DataFrame   → works with _____________ data (has schema)
```

---

**Can you create an RDD from:**
- A Python list? → YES 
- A file on HDFS? → YES 
- A DataFrame? → YES 


In [0]:
# BUILD 1 — Practice
# Create a simple RDD from a Python list

rdd1 = ([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Print number of partitions
print("Partitions:", rdd1.getNumPartitions())

#2
# Collect all elements
print("Data:", rdd1.collect())

# Count elements
print("Count:", rdd1.count())


---
## 📌 BUILD 2 — How Spark Reads Data
*Watch Video 02 first*

### Fill in the blanks:

**When Spark reads a file from HDFS:**

Step 1: File is split into partition of 128 MB each
Step 2: Each split becomes one partition
Step 3: Each partition is processed by one core

---

**Default partition size in Spark:** 128 MB

---

**Spark reads data LAZILY. What does this mean?**

```
Answer:it means until any action item has been it doesnt do any work 
```

---

**Two ways to create an RDD:**

1. `sc.parallelize()` → used for local data file
2. `sc.textFile()` → used for any external data file to convert it


In [0]:
# BUILD 2 — Read a file into RDD
# Create sample data first
data = ["Mumbai,Tools,50000",
        "Delhi,Paint,30000",
        "Mumbai,Garden,20000",
        "Delhi,Tools,45000",
        "Bangalore,Paint,25000"]

rdd_data = spark.sparkContext.parallelize(data)
print("Partitions:", rdd_data.getNumPartitions())
print("First 2 rows:", rdd_data.take(2))
print("Total rows:", rdd_data.count())


---
## 📌 BUILD 3 — Partitioning
*Watch Video 03 first*

### Fill in the blanks:

**What is a partition?**
```
Answer:
```

**More partitions = more _____________ (good or bad?)**

**Less partitions = less _____________ overhead**

---

**When you run parallelize([1,2,3,4,5,6,7,8]):**

Default partitions on local machine = _____________

---

**How to set number of partitions manually:**
```python
rdd = sc.parallelize(data, _____) ← fill the parameter
```

---

**EXPLAIN: Why does partition count affect performance?**
```
Answer:
```


In [0]:
# BUILD 3 — Partition Practice
rdd = sc.parallelize(range(1, 9))  # 8 elements

print("Default partitions:", rdd.getNumPartitions())

# Create with specific partitions
rdd_4 = sc.parallelize(range(1, 9), 4)
print("With 4 partitions:", rdd_4.getNumPartitions())

rdd_2 = sc.parallelize(range(1, 9), 2)
print("With 2 partitions:", rdd_2.getNumPartitions())

# See what's in each partition
print("Partition contents:", rdd_4.glom().collect())


---
## 📌 BUILD 4 — RDD Operations Part 1
*Watch Video 04 first*

### Transformation vs Action — Fill the table:

| Operation | T or A | Runs immediately? | Returns |
|-----------|--------|------------------|---------|
| map()     |        |                  |         |
| filter()  |        |                  |         |
| flatMap() |        |                  |         |
| collect() |        |                  |         |
| count()   |        |                  |         |
| take(n)   |        |                  |         |
| reduce()  |        |                  |         |
| first()   |        |                  |         |

---

**map() vs flatMap() — key difference:**
```
map()     → each input produces _____ output(s)
flatMap() → each input produces _____ output(s)
```

---

**filter() — what does it do?**
```
Answer:
```


In [0]:
# BUILD 4 — map, filter, flatMap Practice

data = ["Mumbai,Tools,50000",
        "Delhi,Paint,30000",
        "Mumbai,Garden,20000",
        "Delhi,Tools,45000",
        "Bangalore,Paint,25000"]

rdd = sc.parallelize(data)

# map — split each row by comma
rdd_split = rdd.map(lambda x: x.split(","))
print("After map (split):", rdd_split.take(3))

# map — extract city only
rdd_city = rdd_split.map(lambda x: x[0])
print("Cities:", rdd_city.collect())

# filter — only Mumbai rows
rdd_mumbai = rdd_split.filter(lambda x: x[0] == "Mumbai")
print("Mumbai only:", rdd_mumbai.collect())

# flatMap — split sentence into words
sentences = sc.parallelize(["hello world", "spark is fast"])
rdd_words = sentences.flatMap(lambda x: x.split(" "))
print("Words:", rdd_words.collect())


---
## 📌 BUILD 5 — RDD Operations Part 2
*Watch Video 05 first*

### Fill in the blanks:

**reduceByKey()** — works on _____________ pairs

**sortBy()** — sorts by _____________ you define

**distinct()** — removes _____________

---

**Key-Value RDD:**
```
(city, revenue) → this is a _____________ pair
Key = city
Value = revenue
```

---

**Practice: What does this produce?**
```python
rdd = sc.parallelize([("A", 10), ("B", 20), ("A", 30), ("B", 5)])
result = rdd.reduceByKey(lambda a, b: a + b)
# result.collect() = ???

Answer:
```


In [0]:
# BUILD 5 — Key-Value RDD Operations

data = ["Mumbai,Tools,50000",
        "Delhi,Paint,30000",
        "Mumbai,Garden,20000",
        "Delhi,Tools,45000",
        "Bangalore,Paint,25000"]

rdd = sc.parallelize(data)
rdd_split = rdd.map(lambda x: x.split(","))

# Create key-value pair: (city, revenue)
rdd_kv = rdd_split.map(lambda x: (x[0], int(x[2])))
print("Key-Value pairs:", rdd_kv.collect())

# reduceByKey — total revenue per city
rdd_total = rdd_kv.reduceByKey(lambda a, b: a + b)
print("Total per city:", rdd_total.collect())

# sortBy — sort by revenue descending
rdd_sorted = rdd_total.sortBy(lambda x: -x[1])
print("Sorted by revenue:", rdd_sorted.collect())

# distinct — unique cities
rdd_cities = rdd_split.map(lambda x: x[0]).distinct()
print("Unique cities:", rdd_cities.collect())


---
## 📌 BUILD 6 — Narrow vs Wide Transformations
*Watch Video 06 first*

### Fill in the blanks:

**Narrow Transformation:**
- Data stays in _____________ partition
- No _____________ across network
- Examples: _____________, _____________, _____________
- Fast or Slow? _____________

**Wide Transformation:**
- Data moves _____________ partitions
- Causes _____________ (data moving across network)
- Examples: _____________, _____________, _____________
- Fast or Slow? _____________

---

**What is a Shuffle? Why is it expensive?**
```
Answer:
```

---

**EXPLAIN: Which causes a shuffle — map() or groupByKey()?**
```
Answer:
```


In [0]:
# BUILD 6 — See Narrow vs Wide in action

rdd = sc.parallelize([("A",1),("B",2),("A",3),("C",1),("B",4)], 2)

# Narrow — map stays in same partition
rdd_mapped = rdd.map(lambda x: (x[0], x[1]*2))
print("map (narrow):", rdd_mapped.collect())

# Wide — groupByKey shuffles data across partitions
rdd_grouped = rdd.groupByKey()
result = rdd_grouped.mapValues(list).collect()
print("groupByKey (wide - shuffle):", result)

# Wide — reduceByKey also shuffles but more efficient
rdd_reduced = rdd.reduceByKey(lambda a,b: a+b)
print("reduceByKey (wide - efficient):", rdd_reduced.collect())


---
## 📌 BUILD 7 — GroupByKey vs ReduceByKey
*Watch Videos 08 + 09 first*

### Key Difference — Fill in:

| | GroupByKey | ReduceByKey |
|--|-----------|------------|
| What it does | Groups all values into a list | |
| Shuffle amount | _____________ | _____________ |
| Memory usage | _____________ | _____________ |
| Use when | Need all values together | |
| Performance | _____________ | _____________ |

---

**WHY is ReduceByKey faster?**
```
Answer:
```

---

**When MUST you use GroupByKey?**
```
Answer:
```


In [0]:
# BUILD 7 — GroupByKey vs ReduceByKey

rdd = sc.parallelize([
    ("Mumbai", 50000), ("Delhi", 30000),
    ("Mumbai", 20000), ("Delhi", 45000),
    ("Mumbai", 15000)
])

# groupByKey — collects ALL values (more memory)
grouped = rdd.groupByKey().mapValues(list).collect()
print("groupByKey result:", grouped)

# reduceByKey — combines values during shuffle (efficient)
reduced = rdd.reduceByKey(lambda a, b: a + b).collect()
print("reduceByKey result:", reduced)

# RULE: If you just want a sum/count/max → use reduceByKey
# RULE: If you need all values together → use groupByKey


---
## 📌 BUILD 8 — Repartition vs Coalesce
*Watch Video 10 first*

### Fill in the blanks:

**repartition(n):**
- Can increase OR decrease partitions? _____________
- Causes shuffle? _____________
- Use when: _____________

**coalesce(n):**
- Can only _____________ partitions (not increase)
- Causes shuffle? _____________
- Use when: _____________

---

**RULE: Which one to use to go from 100 → 10 partitions?**
```
Answer and why:
```

**RULE: Which one to use to go from 5 → 20 partitions?**
```
Answer and why:
```


In [0]:
# BUILD 8 — Repartition vs Coalesce

rdd = sc.parallelize(range(1, 21), 10)
print("Original partitions:", rdd.getNumPartitions())

# repartition — can go up or down (causes full shuffle)
rdd_rep = rdd.repartition(4)
print("After repartition(4):", rdd_rep.getNumPartitions())

rdd_rep2 = rdd.repartition(20)
print("After repartition(20):", rdd_rep2.getNumPartitions())

# coalesce — can only go down (no full shuffle)
rdd_coal = rdd.coalesce(4)
print("After coalesce(4):", rdd_coal.getNumPartitions())

# coalesce cannot go up — it will stay at original
rdd_coal2 = rdd.coalesce(20)
print("coalesce(20) stays at:", rdd_coal2.getNumPartitions())


---
## 📌 BUILD 9 — Full Practice Problems
*Attempt without looking at answers*

### Problem 1 — Word Count (Classic Spark)
Count how many times each word appears in the sentences below.
Expected output: [('spark', 2), ('is', 2), ('fast', 1), ('great', 1), ('fun', 1)]


In [0]:
# Problem 1 — Word Count
sentences = sc.parallelize([
    "spark is fast",
    "spark is great",
    "learning is fun"
])

# YOUR CODE:
# Step 1: flatMap to split into words
# Step 2: map to create (word, 1) pairs
# Step 3: reduceByKey to sum counts
# Step 4: sortBy count descending



### Problem 2 — Revenue by City
From the sales data — find total revenue per city. Sort highest first.

In [0]:
# Problem 2 — Revenue by City
sales = sc.parallelize([
    ("Mumbai", "Tools", 50000),
    ("Delhi", "Paint", 30000),
    ("Mumbai", "Garden", 20000),
    ("Delhi", "Tools", 45000),
    ("Bangalore", "Paint", 25000),
    ("Mumbai", "Tools", 35000),
    ("Delhi", "Garden", 15000),
])

# YOUR CODE:
# Step 1: map to (city, revenue) pairs
# Step 2: reduceByKey to sum revenue per city
# Step 3: sortBy revenue descending



### Problem 3 — Filter + Count
From the sales data — how many sales transactions are from Mumbai?

In [0]:
# Problem 3 — Filter and Count
# Use the sales rdd from Problem 2

# YOUR CODE:



---
## ✅ ANSWER KEY — Run after attempting

In [0]:
# ANSWER — Problem 1: Word Count
sentences = sc.parallelize(["spark is fast","spark is great","learning is fun"])

result = (sentences
    .flatMap(lambda x: x.split(" "))
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: -x[1]))

print("Word count:", result.collect())


In [0]:
# ANSWER — Problem 2: Revenue by City
sales = sc.parallelize([
    ("Mumbai","Tools",50000),("Delhi","Paint",30000),
    ("Mumbai","Garden",20000),("Delhi","Tools",45000),
    ("Bangalore","Paint",25000),("Mumbai","Tools",35000),
    ("Delhi","Garden",15000),
])

result = (sales
    .map(lambda x: (x[0], x[2]))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: -x[1]))

print("Revenue by city:", result.collect())


In [0]:
# ANSWER — Problem 3: Mumbai count
sales = sc.parallelize([
    ("Mumbai","Tools",50000),("Delhi","Paint",30000),
    ("Mumbai","Garden",20000),("Delhi","Tools",45000),
    ("Bangalore","Paint",25000),("Mumbai","Tools",35000),
    ("Delhi","Garden",15000),
])

mumbai_count = sales.filter(lambda x: x[0] == "Mumbai").count()
print("Mumbai transactions:", mumbai_count)


---
## 📝 EXPLAIN BLOCK — Answer in your own words

**Q1: What is an RDD and what makes it different from a DataFrame?**
```
Your answer:
```

**Q2: What is the difference between a Transformation and an Action?**
```
Your answer:
```

**Q3: Why is reduceByKey better than groupByKey for aggregations?**
```
Your answer:
```

**Q4: When would you use repartition vs coalesce?**
```
Your answer:
```

**Q5: What is a shuffle and why is it expensive?**
```
Your answer:
```

**Q6: What is the difference between narrow and wide transformations?**
```
Your answer:
```
